In [ ]:
import pandas as pd
import geopandas as gpd

from geopy.geocoders import Nominatim
from ipyleaflet import Map, GeoData, basemaps, LayersControl

geolocator = Nominatim(user_agent="Python-kurs-2024")

In [ ]:
# Read csv without headers, set column names manually.
df = pd.read_csv(
    '../data/adresser.txt',
    header=None,
    names=['adresse', 'postnr_sted', 'land']
)

In [ ]:
# Invoke geocoder on 'adresse' and 'postnr/sted'
df['geocode_res'] = df.apply(
    lambda x: geolocator.geocode(x['adresse'] + ', ' + x['postnr_sted']),
    axis=1
)

In [ ]:
# Fetch lat/long from geocoder result
df['lat'] = df['geocode_res'].apply(lambda x: x.latitude)
df['lon'] = df['geocode_res'].apply(lambda x: x.longitude)

In [ ]:
# Convert dataframe into geodataframe, converting latlon into proper points.
gdf = gpd.GeoDataFrame(
    df[['adresse', 'postnr_sted', 'land']],
    geometry=gpd.points_from_xy(df['lon'], df['lat'])
)

In [ ]:
m = Map(
    basemap=basemaps.OpenStreetMap.Mapnik,
    center=(59.91228, 10.78560),
    zoom=10
)

geo_data = GeoData(geo_dataframe = gdf,
    name = 'addresser')

m.add(geo_data)
m.add(LayersControl())

m